In [1]:
# Test OPENAI connection:
# uv add requests minsearch openai jupyter python-dotenv
# This installs:
# requests - to fetch the FAQ dataset from the internet
# minsearch - a simple in-memory search engine for indexing and searching text
# openai - the OpenAI API client for calling the LLM
# jupyter - the notebook environment where we'll write and run code
# python-dotenv - to load API keys from a .env file

# Check that the OpenAI client works:

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [4]:
openai_client
# should be something like <openai.OpenAI at 0x18e6f7e02f0>

In [5]:
# define a function to talk to the LLM:

def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [6]:
# test it:

llm("Hey, what's up?")

'Hey! Not much — how can I help?'

In [7]:
# if no prior context exist, then pure hallucination - like this course-specific question:

question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Absolutely — if the course is still open, you should be able to join now.

A couple of quick checks:
- **Is enrollment still open?** Some courses allow late sign-ups, while others have a cutoff date.
- **Will you miss anything important?** If the course has already started, you may want to catch up on the first materials.
- **Do you need permission?** Some courses require approval from the instructor or admin team.

If you want, I can help you draft a short message to the course organizer asking whether late enrollment is still possible.


In [8]:
# The LLM gives a generic answer. It might say "you can usually join" or "check the course website." 
# It doesn't know about our specific Zoomcamp courses, their enrollment policies, or their schedules. 
# It tries to be helpful, but has no idea about actual enrollment status or policies.

# This is different from a question like "how do I cook salmon?" - the LLM knows the answer because 
# cooking salmon is common knowledge. But our courses are not in the training data.

question = "how do I cook salmon?"
answer = llm(question)
print(answer)


Here are a few easy ways to cook salmon, depending on what you have:

## 1) Pan-seared salmon
**Best for:** crispy skin, quick cooking

**You need:**
- Salmon fillets
- Salt and pepper
- Oil or butter
- Optional: lemon, garlic, herbs

**Steps:**
1. Pat the salmon dry with paper towels.
2. Season both sides with salt and pepper.
3. Heat a skillet over medium-high heat with a little oil.
4. Put salmon in the pan **skin-side down** if it has skin.
5. Cook about **4–6 minutes** without moving it.
6. Flip and cook **1–3 minutes** more, until it flakes easily.

## 2) Baked salmon
**Best for:** easiest method

**Steps:**
1. Preheat oven to **400°F / 200°C**.
2. Put salmon on a lined baking sheet or in a dish.
3. Season with salt, pepper, oil, lemon, etc.
4. Bake **10–15 minutes**, depending on thickness.

## 3) Air fryer salmon
**Best for:** fast and hands-off

1. Preheat air fryer to **390–400°F (200°C)**.
2. Season salmon.
3. Cook **7–10 minutes**.

## How to tell it’s done
- It should **fl

In [9]:
# More context can fix this. The FAQ website has questions and answers about our courses.

# Copy some of that content into the prompt:

context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""


In [11]:
question = "I just discovered the course. Can I join now?"

# moving back to course

In [13]:
# Build a prompt that includes both the question and the context:

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""
prompt

'\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\nI just discovered the course. Can I join now?\n\nContext:\n\nI just discovered the course. Can I still join?\nYes, but if you want to receive a certificate, you need to submit your project while we\'re still accepting submissions.\n\nCourse: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nYou don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nWhat is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?\nThe zoom link is only published to instructors/present

In [14]:
# Instead of sending the raw question to the LLM, we send this prompt:

answer = llm(prompt)
print(answer)

# Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.
# This is RAG - providing model a context to ground it down and reduce hallucinations


Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.


In [15]:
# RAG stands for Retrieval-Augmented Generation. Generation is the LLM producing text, and retrieval is search. 
# We retrieve relevant documents from our knowledge base and use them to augment what the LLM generates. 
# That search step is what gives the LLM the context it needs to answer correctly.

# What we just did was naive. I knew in advance which FAQ entry held the answer and pasted it in by hand. 
# What we want instead is to perform search automatically. We take the student's question, 
# find the most relevant documents, and send those to the LLM.

# In code, it looks like this:

def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [16]:
# The LLM only sees the documents we hand it, so its answers are grounded in our data. 
# If the right document is retrieved, the answer is good. 
# If it's not, the LLM gets the wrong context and the answer is wrong. 
# Your model is only as good as your retrieval, so search quality matters a lot for RAG.

# The database and the LLM can be anything. 
# In this course we use minsearch and then sqlitesearch for search, and OpenAI for the LLM. 
# But you can swap any component for another and see what works better.

# The FAQ data is available as JSON from the DataTalks.Club website. 
# I maintain that site, so I made the data available at a JSON endpoint we can fetch directly.

# Let's fetch it:

import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [20]:
courses_raw[:10]

[{'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255}]

In [22]:
# This returns a list of courses. Each course has a path field that points to its FAQ data.

# Let's fetch all the FAQ documents from all courses:

documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""
    print(course_url)

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

https://datatalks.club/faq/json/machine-learning-zoomcamp.json
https://datatalks.club/faq/json/llm-zoomcamp.json
https://datatalks.club/faq/json/data-engineering-zoomcamp.json
https://datatalks.club/faq/json/mlops-zoomcamp.json


1208

In [23]:
# Each entry has:

# id - unique identifier
# course - course slug (e.g., machine-learning-zoomcamp)
# section - which section of the course
# question - the FAQ question
# answer - the FAQ answer
# Let's look at one:

documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [ ]:
# In the RAG pipeline, this dataset is our knowledge base:

# We index all the documents (the search step)
# When a student asks a question, we search the index
# The search returns the most relevant FAQ entries
# We give those entries to the LLM as context
# The LLM generates an answer based on the context
# The question and answer fields contain the text we'll search through. 
# The course field lets us filter by course. For example, if a student asks 
# about the data engineering course, we skip results from the ML course. 
# The section field helps with ranking - knowing which part of the course a question belongs to is useful context.

